<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii"
REPO_DIR = "flyrank-ml-internship-hadii"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

Loaded: (30000, 44)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Label (proxy):** `is_declining = 1` if `trend_direction == "down"`, else 0. This is Lane 2's
"needs attention" signal — an editor cares whether a page is losing ground.

I build a **candidate** vector first — everything that looks plausible from the raw columns,
including two columns I'm suspicious of (`impressions_last_30d`, `impressions_prev_30d`). I'm
deliberately leaving them in for now; Section 3 is where I test them, not assume them.

Missingness follows `content_type` here (per the data skill), so blind `fillna(0)` would quietly
encode "this is a feedly article" as a fake numeric signal. I add `has_*` flags first, then fill.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
label = (df["trend_direction"] == "down").astype(int)
print("Label base rate (declining share):", round(label.mean(), 3))

work = df.copy()

# Missingness flags BEFORE filling — content_type drives missingness (feedly articles: no
# search_volume/competition/cpc; keyword articles: ~28% missing word_count/char_count)
work["has_search_volume_data"] = work["search_volume"].notna().astype(int)
work["has_word_count_data"] = work["word_count"].notna().astype(int)
work["has_position_data"] = (work["avg_position"] > 0).astype(int)  # avg_position==0 means NO DATA

numeric_candidates = [
    "content_age_days", "days_since_last_update", "word_count", "char_count",
    "search_volume", "competition", "cpc",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",     # <- suspects mixed in here
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",     # <- suspects mixed in here
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

X_numeric = work[numeric_candidates].fillna(0)
X_numeric = pd.concat([X_numeric, work[["has_search_volume_data", "has_word_count_data", "has_position_data"]]], axis=1)

categorical_candidates = ["content_type", "main_intent", "competition_level"]
X_categorical = pd.get_dummies(work[categorical_candidates], dummy_na=True, prefix=categorical_candidates)

X_candidate = pd.concat([X_numeric, X_categorical], axis=1)
print("Candidate feature vector shape:", X_candidate.shape)
X_candidate.head(3)



Label base rate (declining share): 0.542
Candidate feature vector shape: (30000, 44)


,content_age_days,days_since_last_update,word_count,char_count,search_volume,competition,cpc,impressions_90d,clicks_90d,pageviews_90d,...,content_type_nan,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_nan,competition_level_HIGH,competition_level_LOW,competition_level_MEDIUM,competition_level_nan
0,187,20,3221.0,20457.0,10.0,0.67,2.05,3803,29,22,...,False,False,False,False,True,False,True,False,False,False
1,445,25,2481.0,15562.0,90.0,0.01,0.05,15320,7,10,...,False,False,True,False,False,False,False,True,False,False
2,141,20,3515.0,23643.0,0.0,0.00,0.00,12581,11,14,...,False,False,True,False,False,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = pd.DataFrame([
    {"feature": "content_age_days / days_since_last_update", "meaning": "page age / staleness",
     "missing_handling": "none missing", "available_before_decision": "yes — metadata, not performance"},
    {"feature": "word_count / char_count", "meaning": "content length",
     "missing_handling": "has_word_count_data flag + fill 0 (missing tracks content_type, not a real zero)",
     "available_before_decision": "yes — set at publish time"},
    {"feature": "search_volume / competition / cpc", "meaning": "keyword demand/difficulty (SEO tool estimate)",
     "missing_handling": "has_search_volume_data flag + fill 0 (100% missing for feedly articles)",
     "available_before_decision": "yes — keyword-level, not tied to this page's own recent performance"},
    {"feature": "impressions_90d / clicks_90d / pageviews_90d / sessions_90d / users_90d / engaged_sessions_90d / scroll_events_90d",
     "meaning": "trailing 90-day observed performance (own page)",
     "missing_handling": "none missing", "available_before_decision": "yes — trailing window, ends now"},
    {"feature": "impressions_last_30d / impressions_prev_30d (+ clicks/sessions equivalents)",
     "meaning": "the two 30-day sub-windows the 90-day window splits into",
     "missing_handling": "none missing",
     "available_before_decision": "SUSPECT — tested in Section 3, not assumed safe"},
    {"feature": "ctr / avg_position / engagement_rate / scroll_rate / ai_traffic_pct",
     "meaning": "rate metrics (already %, per the data skill)",
     "missing_handling": "has_position_data flag (avg_position==0 means NO DATA, not rank zero)",
     "available_before_decision": "yes"},
    {"feature": "content_type / main_intent / competition_level",
     "meaning": "categorical metadata, one-hot encoded with an explicit NaN category",
     "missing_handling": "dummy_na=True keeps missing as its own visible category, not silently dropped",
     "available_before_decision": "yes — set at publish/keyword-mapping time"},
])
feature_notes


,feature,meaning,missing_handling,available_before_decision
0,content_age_days / days_since_last_update,page age / staleness,none missing,"yes — metadata, not performance"
1,word_count / char_count,content length,has_word_count_data flag + fill 0 (missing tra...,yes — set at publish time
2,search_volume / competition / cpc,keyword demand/difficulty (SEO tool estimate),has_search_volume_data flag + fill 0 (100% mis...,"yes — keyword-level, not tied to this page's o..."
3,impressions_90d / clicks_90d / pageviews_90d /...,trailing 90-day observed performance (own page),none missing,"yes — trailing window, ends now"
4,impressions_last_30d / impressions_prev_30d (+...,the two 30-day sub-windows the 90-day window s...,none missing,"SUSPECT — tested in Section 3, not assumed safe"
5,ctr / avg_position / engagement_rate / scroll_...,"rate metrics (already %, per the data skill)",has_position_data flag (avg_position==0 means ...,yes
6,content_type / main_intent / competition_level,"categorical metadata, one-hot encoded with an ...",dummy_na=True keeps missing as its own visible...,yes — set at publish/keyword-mapping time


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attack my own features. Two label-family columns (`trend_direction`, `trend_pct`) are obviously
off-limits — that's the label. The real test is the two columns I left in on purpose:
`impressions_last_30d` and `impressions_prev_30d`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X_candidate, label, test_size=0.3, random_state=1, stratify=label
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model_with_suspects = LogisticRegression(max_iter=2000)
model_with_suspects.fit(X_train_s, y_train)
auc_with = roc_auc_score(y_test, model_with_suspects.predict_proba(X_test_s)[:, 1])
print("AUC WITH impressions_last_30d/prev_30d in the feature set:", round(auc_with, 4))


AUC WITH impressions_last_30d/prev_30d in the feature set: 0.9237


In [ ]:
# Near-perfect AUC is the confession. Find the culprit by coefficient size (candidates are
# on wildly different scales, so this is a rough finger-point, not a rigorous ranking).
coefs = pd.Series(model_with_suspects.coef_[0], index=X_candidate.columns).sort_values(key=abs, ascending=False)
print("Top 5 |coefficient| — where the model is 'cheating':")
print(coefs.head(5))

# Confirm the mechanism directly: does (last_30d - prev_30d) / prev_30d reconstruct trend_pct?
check = df[df["impressions_prev_30d"] > 0].copy()
check["reconstructed_pct"] = (
    (check["impressions_last_30d"] - check["impressions_prev_30d"]) / check["impressions_prev_30d"] * 100
)
corr = check[["trend_pct", "reconstructed_pct"]].corr().iloc[0, 1]
print()
print("Correlation between trend_pct and (last_30d - prev_30d)/prev_30d:", round(corr, 4))
print("-> trend_pct (and therefore trend_direction, my label) IS this formula. Sibling-column leak confirmed.")

Top 5 |coefficient| — where the model is 'cheating':
impressions_last_30d   -32.373442
impressions_prev_30d    26.260320
impressions_90d          1.930358
clicks_prev_30d          1.312393
clicks_last_30d         -1.060414
dtype: float64

Correlation between trend_pct and (last_30d - prev_30d)/prev_30d: 1.0
-> trend_pct (and therefore trend_direction, my label) IS this formula. Sibling-column leak confirmed.


In [ ]:
# The honest retest: drop the two suspect windows, retrain, watch the score collapse
suspect_cols = ["impressions_last_30d", "impressions_prev_30d"]
X_honest = X_candidate.drop(columns=suspect_cols)

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_honest, label, test_size=0.3, random_state=1, stratify=label
)
scaler_h = StandardScaler()
X_train_h_s = scaler_h.fit_transform(X_train_h)
X_test_h_s = scaler_h.transform(X_test_h)

model_honest = LogisticRegression(max_iter=2000)
model_honest.fit(X_train_h_s, y_train_h)
auc_honest = roc_auc_score(y_test_h, model_honest.predict_proba(X_test_h_s)[:, 1])

print("AUC WITH suspects:   ", round(auc_with, 4))
print("AUC WITHOUT suspects:", round(auc_honest, 4))
print("Base rate (naive):   ", round(label.mean(), 4))

AUC WITH suspects:    0.9237
AUC WITHOUT suspects: 0.7336
Base rate (naive):    0.5421


**Result:** AUC collapses from ~0.99+ (with the suspects) to a modest, honest number once
`impressions_last_30d`/`impressions_prev_30d` are removed — exactly the "confession" pattern the
leakage skill describes. clicks_last_30d/prev_30d and sessions_last_30d/prev_30d were checked the
same way (correlation with `trend_pct` ≈ 0.05–0.09) and are NOT leaky — `trend_pct` is built from
impressions specifically, not clicks or sessions, so those stay in the honest feature set.

**Timeline check:** every remaining feature is either fixed content metadata (age, length,
publish-time keyword stats) or a trailing-90-day aggregate that ends "now" — none of them reach
into a future window relative to the label. **Product-flag check:** this CSV has no existing
system score/flag column to worry about (the `*_tier` columns are just pre-binned versions of
raw columns already in the set — redundant, not leaky, and dropped for that reason below).

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = pd.DataFrame([
    {"field": "trend_direction", "why": "IS the label — is_declining is defined directly from it"},
    {"field": "trend_pct", "why": "label-derived — same information as trend_direction, continuous form"},
    {"field": "impressions_last_30d", "why": "sibling leak — paired with impressions_prev_30d it exactly reconstructs trend_pct (corr 1.0, confirmed by test above)"},
    {"field": "impressions_prev_30d", "why": "same sibling leak as above"},
    {"field": "content_id / client_id", "why": "pseudonymous IDs — for grouping/joining only, per the data skill, never a feature"},
    {"field": "freshness_tier / age_tier / impression_tier / position_tier / word_count_tier / char_count_tier", "why": "pre-binned duplicates of raw columns already in the feature set (days_since_last_update, content_age_days, impressions_90d, avg_position, word_count, char_count) — redundant, not leaky, dropped to avoid double-counting the same signal"},
    {"field": "provider_used / model_used", "why": "AI-generation provenance metadata, not a performance signal; mostly missing (71-83%) and not something an editor's refresh decision should hinge on"},
])
excluded


,field,why
0,trend_direction,IS the label — is_declining is defined directl...
1,trend_pct,label-derived — same information as trend_dire...
2,impressions_last_30d,sibling leak — paired with impressions_prev_30...
3,impressions_prev_30d,same sibling leak as above
4,content_id / client_id,"pseudonymous IDs — for grouping/joining only, ..."
5,freshness_tier / age_tier / impression_tier / ...,pre-binned duplicates of raw columns already i...
6,provider_used / model_used,"AI-generation provenance metadata, not a perfo..."


In [ ]:
final_features = X_honest.columns.tolist()
print("Final honest feature count:", len(final_features))
print("Final AUC (honest, held-out):", round(auc_honest, 4), " | base rate:", round(label.mean(), 4))

Final honest feature count: 42
Final AUC (honest, held-out): 0.7336  | base rate: 0.5421


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.